# HeadsUp - Feature Engineering

**Input:** `train.csv`, `val.csv`, `test.csv` from Stage 3 (65,752 orders, chronologically split)
**Output:** `model_ready_train.csv`, `model_ready_val.csv`, `model_ready_test.csv` - fully encoded, scaled feature matrices ready(baseline and model development)

---

## Purpose of this notebook

This is the consolidated Stage 4 deliverable. It takes the clean, aggregated order level data from Stage 3 and works through five steps to produce the final modelling ready feature matrices:

1. **Temporal features** - hour, day of week, month, weekend/holiday flags
2. **Shipping & order context features** - resolves the Shipping Mode multicollinearity flag from EDA, builds `sales_per_scheduled_day` and `is_express_shipping`
3. **Geographic target encoding** - country and region delay rates, fit strictly on training data
4. **Categorical encoding** - one-hot for low cardinality fields, frequency encoding for `Category Name`
5. **Final matrix assembly** - drops superseded raw columns, scales numeric features, saves the model ready sets



## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

TRAIN_IN = Path('../data/processed/train.csv')
VAL_IN = Path('../data/processed/val.csv')
TEST_IN = Path('../data/processed/test.csv')

PROCESSED_DIR = Path('../data/processed')

train_df = pd.read_csv(TRAIN_IN)
val_df = pd.read_csv(VAL_IN)
test_df = pd.read_csv(TEST_IN)

print(f"Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")


Train: (46026, 21), Val: (7890, 21), Test: (11836, 21)


## 1. Temporal features

EDA found day of week and month patterns were essentially flat (weak standalone signal). We still engineer these, since a weak *individual* signal doesn't rule out the model picking up a useful interaction effect, but we shouldn't expect these to move the needle much on their own.


In [2]:
def add_temporal_features(df):
    df = df.copy()
    df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'], errors='coerce')
    df['order_hour'] = df['order date (DateOrders)'].dt.hour
    df['order_dayofweek'] = df['order date (DateOrders)'].dt.dayofweek
    df['order_month'] = df['order date (DateOrders)'].dt.month
    df['order_is_weekend'] = (df['order_dayofweek'] >= 5).astype(int)
    df['order_is_holiday_season'] = df['order_month'].isin([11, 12]).astype(int)
    return df

train_df = add_temporal_features(train_df)
val_df = add_temporal_features(val_df)
test_df = add_temporal_features(test_df)

for col in ['order_dayofweek', 'order_month', 'order_is_weekend', 'order_is_holiday_season']:
    rate = train_df.groupby(col)['Late_delivery_risk'].mean()
    print(f"{col}: late rate spread {rate.max() - rate.min():.3f}")


order_dayofweek: late rate spread 0.015
order_month: late rate spread 0.024
order_is_weekend: late rate spread 0.005
order_is_holiday_season: late rate spread 0.001


**What we found:** confirmed, exactly consistent with EDA. All four spreads are under 2.5 percentage points (day of week 1.5pp, month 2.4pp, weekend 0.5pp, holiday season 0.1pp) - tiny compared to Shipping Mode's 57pp spread. Kept anyway for potential interaction effects, but not expected to rank as important features on their own.


## 2. Shipping & order context features

EDA flagged a potential multicollinearity risk between `Shipping Mode` and `Days for shipment (scheduled)`. We check this directly before deciding what to do with either column.


In [3]:
crosstab = pd.crosstab(train_df['Shipping Mode'], train_df['Days for shipment (scheduled)'])
crosstab


Days for shipment (scheduled),0,1,2,4
Shipping Mode,,,,
First Class,0,7049,0,0
Same Day,2478,0,0,0
Second Class,0,0,8929,0
Standard Class,0,0,0,27570


**What we found:** the relationship is a **perfect 1:1 mapping**, not just near deterministic - First Class↔1 day, Same Day↔0 days, Second Class↔2 days, Standard Class↔4 days, with zero exceptions across the entire training set. `Days for shipment (scheduled)` carries no information beyond what `Shipping Mode` already encodes, and keeping both would destabilize Logistic Regression's coefficients - a real problem given interpretability is the whole reason we're using it as a baseline.

**Decision: drop `Days for shipment (scheduled)`, but only after using it to build `sales_per_scheduled_day` below.**


In [4]:
def add_shipping_features(df):
    df = df.copy()
    df['sales_per_scheduled_day'] = df['Sales'] / df['Days for shipment (scheduled)'].replace(0, np.nan)
    df['sales_per_scheduled_day'] = df['sales_per_scheduled_day'].fillna(df['Sales'])

    df['is_express_shipping'] = df['Shipping Mode'].isin(['First Class', 'Same Day']).astype(int)
    return df

train_df = add_shipping_features(train_df)
val_df = add_shipping_features(val_df)
test_df = add_shipping_features(test_df)

print("Express shipping late rate (train):")
print(train_df.groupby('is_express_shipping')['Late_delivery_risk'].mean())

train_df = train_df.drop(columns=['Days for shipment (scheduled)'])
val_df = val_df.drop(columns=['Days for shipment (scheduled)'])
test_df = test_df.drop(columns=['Days for shipment (scheduled)'])
print("\nDropped Days for shipment (scheduled). Confirming engineered features survived:")
print([c for c in train_df.columns if c in ['sales_per_scheduled_day', 'is_express_shipping']])


Express shipping late rate (train):
is_express_shipping
0    0.477027
1    0.822085
Name: Late_delivery_risk, dtype: float64

Dropped Days for shipment (scheduled). Confirming engineered features survived:
['sales_per_scheduled_day', 'is_express_shipping']


**What we found:** a large, meaningful gap - 47.70% late rate for non express orders vs. **82.21%** for express orders (First Class + Same Day combined), consistent with EDA's finding that these two modes individually sit well above the other two.


## 3. Geographic target encoding

EDA found `Order Country` carries a much wider late rate spread (~30pp) than `Order Region` (~9pp). We target encode both, fitting strictly on the training set only.

**Leakage rule:** the encoding map is fit only on training data, then applied to validation and test unchanged. Fitting on the full dataset before splitting would leak future information into the encoding.


In [5]:
global_mean = train_df['Late_delivery_risk'].mean()

country_encoding_map = train_df.groupby('Order Country')['Late_delivery_risk'].mean()
region_encoding_map = train_df.groupby('Order Region')['Late_delivery_risk'].mean()

def apply_target_encoding(df, mapping, source_col, new_col, fallback):
    df = df.copy()
    df[new_col] = df[source_col].map(mapping).fillna(fallback)
    return df

for name, df_ref in [('train', train_df), ('val', val_df), ('test', test_df)]:
    n_unseen = (~df_ref['Order Country'].isin(country_encoding_map.index)).sum()
    print(f"{name}: {n_unseen} rows with an Order Country unseen in training")

for col_src, col_new, mapping in [('Order Country', 'country_delay_rate', country_encoding_map),
                                    ('Order Region', 'region_delay_rate', region_encoding_map)]:
    train_df = apply_target_encoding(train_df, mapping, col_src, col_new, global_mean)
    val_df = apply_target_encoding(val_df, mapping, col_src, col_new, global_mean)
    test_df = apply_target_encoding(test_df, mapping, col_src, col_new, global_mean)


train: 0 rows with an Order Country unseen in training
val: 0 rows with an Order Country unseen in training
test: 0 rows with an Order Country unseen in training


**What we found:** full coverage, zero fallback needed. All 164 countries in training appear across validation and test with no unseen categories - a good confirmation that the chronological split preserved full geographic coverage despite being ordered by time rather than randomly shuffled.


## 4. Remaining categorical encoding

One hot encoding for low cardinality fields (`Shipping Mode`, `Customer Segment`, `Type`), and frequency encoding for `Category Name` - confirmed reliable in Stage 3 after resolving the department scoping question.


In [6]:
onehot_cols = ['Shipping Mode', 'Customer Segment', 'Type']

train_df = pd.get_dummies(train_df, columns=onehot_cols, prefix=onehot_cols, drop_first=False)
val_df = pd.get_dummies(val_df, columns=onehot_cols, prefix=onehot_cols, drop_first=False)
test_df = pd.get_dummies(test_df, columns=onehot_cols, prefix=onehot_cols, drop_first=False)

train_cols = train_df.columns
val_df = val_df.reindex(columns=train_cols, fill_value=0)
test_df = test_df.reindex(columns=train_cols, fill_value=0)

category_freq_map = train_df['Category Name'].value_counts(normalize=True) if 'Category Name' in train_df.columns else None


In [7]:
category_freq_map = train_df['Category Name'].value_counts(normalize=True)
global_freq_fallback = category_freq_map.min()

train_df['category_frequency'] = train_df['Category Name'].map(category_freq_map).fillna(global_freq_fallback)
val_df['category_frequency'] = val_df['Category Name'].map(category_freq_map).fillna(global_freq_fallback)
test_df['category_frequency'] = test_df['Category Name'].map(category_freq_map).fillna(global_freq_fallback)

print(f"Shape after all categorical encoding: {train_df.shape}")


Shape after all categorical encoding: (46026, 38)


**What we found:** column alignment worked cleanly with no errors - 4 (Shipping Mode) + 3 (Customer Segment) + 4 (Type) = 11 dummy columns, matching expectations exactly. No category turned out missing from any split, consistent with the full coverage pattern also found for country in Section 3.


## 5. Final feature matrix assembly

Drops raw columns now superseded by their engineered/encoded versions, scales numeric features (fit on training data only), and saves the final modelling ready sets.


In [8]:
columns_to_drop = [
    'Order Id', 'Order Country', 'Order Region', 'Order City', 'Order State',
    'Category Name', 'Category Id', 'order date (DateOrders)', 'Delivery Status',
]
columns_to_drop = [c for c in columns_to_drop if c in train_df.columns]

train_df = train_df.drop(columns=columns_to_drop)
val_df = val_df.drop(columns=columns_to_drop)
test_df = test_df.drop(columns=columns_to_drop)

print(f"Dropped: {columns_to_drop}")


Dropped: ['Order Id', 'Order Country', 'Order Region', 'Order City', 'Order State', 'Category Name', 'Category Id', 'order date (DateOrders)', 'Delivery Status']


In [9]:
numeric_cols_to_scale = [
    'Sales', 'Order Item Quantity', 'Benefit_per_order_capped', 'n_line_items',
    'n_distinct_categories', 'sales_per_scheduled_day',
    'country_delay_rate', 'region_delay_rate', 'category_frequency', 'priority_value_component',
]
numeric_cols_to_scale = [c for c in numeric_cols_to_scale if c in train_df.columns]

scaler = StandardScaler()
train_df[numeric_cols_to_scale] = scaler.fit_transform(train_df[numeric_cols_to_scale])
val_df[numeric_cols_to_scale] = scaler.transform(val_df[numeric_cols_to_scale])
test_df[numeric_cols_to_scale] = scaler.transform(test_df[numeric_cols_to_scale])

print(f"Scaled columns: {numeric_cols_to_scale}")


Scaled columns: ['Sales', 'Order Item Quantity', 'Benefit_per_order_capped', 'n_line_items', 'n_distinct_categories', 'sales_per_scheduled_day', 'country_delay_rate', 'region_delay_rate', 'category_frequency', 'priority_value_component']


**Note:** `priority_value_component` is scaled here for modelling consistency, but a separate **unscaled copy** is kept for the Streamlit app's priority ranking display later - a standardized value (which can be negative) isn't meaningful to show an ops user as "order value."


In [10]:
unscaled_priority = pd.concat([
    train_df[['priority_value_component']].assign(split='train'),
    val_df[['priority_value_component']].assign(split='val'),
    test_df[['priority_value_component']].assign(split='test'),
]).reset_index(drop=True)


In [11]:
assert train_df.isnull().sum().sum() == 0, "Missing values found in final train set!"
assert val_df.isnull().sum().sum() == 0, "Missing values found in final val set!"
assert test_df.isnull().sum().sum() == 0, "Missing values found in final test set!"

print(f"Final shapes — Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")
print(f"\nFinal feature columns (excluding target):")
print([c for c in train_df.columns if c != 'Late_delivery_risk'])

train_df.to_csv(PROCESSED_DIR / 'model_ready_train.csv', index=False)
val_df.to_csv(PROCESSED_DIR / 'model_ready_val.csv', index=False)
test_df.to_csv(PROCESSED_DIR / 'model_ready_test.csv', index=False)
unscaled_priority.to_csv(PROCESSED_DIR / 'priority_value_unscaled.csv', index=False)

print("\nSaved model-ready train/val/test sets and the unscaled priority-value reference.")


Final shapes — Train: (46026, 29), Val: (7890, 29), Test: (11836, 29)

Final feature columns (excluding target):
['Sales', 'Order Item Quantity', 'Benefit per order', 'n_line_items', 'n_distinct_categories', 'Benefit_per_order_capped', 'priority_value_component', 'order_hour', 'order_dayofweek', 'order_month', 'order_is_weekend', 'order_is_holiday_season', 'sales_per_scheduled_day', 'is_express_shipping', 'country_delay_rate', 'region_delay_rate', 'Shipping Mode_First Class', 'Shipping Mode_Same Day', 'Shipping Mode_Second Class', 'Shipping Mode_Standard Class', 'Customer Segment_Consumer', 'Customer Segment_Corporate', 'Customer Segment_Home Office', 'Type_CASH', 'Type_DEBIT', 'Type_PAYMENT', 'Type_TRANSFER', 'category_frequency']

Saved model-ready train/val/test sets and the unscaled priority-value reference.


**What we found:** final shapes confirmed at Train (46,026, 29), Validation (7,890, 29), Test (11,836, 29) - 28 features plus the target, no missing values in any split. All engineered features from every earlier section (including `sales_per_scheduled_day` and `is_express_shipping`) are present and accounted for.

---

## Summary 

| # | Decision | Reasoning |
|---|---|---|
| 1 | Temporal features kept despite weak individual signal | Interaction effects possible even when standalone spread is small (all under 2.5pp) |
| 2 | `Days for shipment (scheduled)` dropped after building `sales_per_scheduled_day` from it | Perfect 1:1 collinearity with `Shipping Mode` confirmed - keeping both would destabilize Logistic Regression coefficients |
| 3 | Country/region delay rate target encoded, fit strictly on training data | Standard leakage prevention practice; confirmed zero unseen category fallback needed |
| 4 | One hot for `Shipping Mode`/`Customer Segment`/`Type`, frequency encoding for `Category Name` | Low cardinality fields suit one hot; `Category Name` confirmed reliable in Stage 3 |
| 5 | Numeric features scaled with `StandardScaler` fit on training data only; unscaled priority value reference kept separately | Needed for Logistic Regression comparability; app display needs real, interpretable order values, not standardized ones |

**Final 27 feature set:** order value/quantity/profit fields, five temporal features, two shipping-context features, two geographic target encoded rates, 11 one-hot encoded categorical columns, and the category frequency encoding.

